<a href="https://colab.research.google.com/github/Jalilnkh/PyTorch-with-Examples-2024/blob/parts/train_en_azb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [8]:
from datasets import load_dataset

dataset = load_dataset("Kartal-Ol/en-azb-548k")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

train.parquet:   0%|          | 0.00/87.9M [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/548900 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [4]:
from transformers import AutoTokenizer

# Load the Arabic tokenizer
tokenizer = AutoTokenizer.from_pretrained("asafaya/bert-base-arabic")

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/334k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [10]:
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/fine_tuned_nmt_model")

In [11]:
# Tokenize function for batched tokenization
def tokenize_function(examples):
    # Extract 'en' and 'azb' for each example in the batch
    source_texts = [example['en'] for example in examples['translation']]
    target_texts = [example['azb'] for example in examples['translation']]

    # Tokenize both source (English) and target (Azerbaijani Arabic script)
    source = tokenizer(source_texts, padding="max_length", truncation=True, max_length=128)
    target = tokenizer(target_texts, padding="max_length", truncation=True, max_length=128)

    return {
        'input_ids': source['input_ids'],
        'attention_mask': source['attention_mask'],
        'labels': target['input_ids']
    }

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/548900 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [6]:
from transformers import T5ForConditionalGeneration

# Load T5 model
model = T5ForConditionalGeneration.from_pretrained("t5-small")


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [19]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",          # Where to save model checkpoints
    evaluation_strategy="epoch",    # Evaluate at the end of every epoch
    learning_rate=2e-5,             # Learning rate
    per_device_train_batch_size=64, # Batch size per GPU
    per_device_eval_batch_size=64,  # Batch size for validation
    num_train_epochs=2,             # Number of epochs
    weight_decay=0.01,              # Weight decay for regularization
    save_total_limit=3,
    report_to="none" # Keep only the last 3 checkpoints
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [15]:
!pip install wandb


In [9]:
!export WANDB_MODE=disabled


In [21]:
from transformers import Trainer
import os

# Create the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

# Start training
trainer.train()

# 9. Save
# Save the model and tokenizer to Google Drive
model.save_pretrained('/content/drive/MyDrive/fine_tuned_nmt_model')
tokenizer.save_pretrained('/content/drive/MyDrive/fine_tuned_nmt_model')

Epoch,Training Loss,Validation Loss
1,1.829400,3.340679
2,1.692200,3.153324


('/content/drive/MyDrive/fine_tuned_nmt_model/tokenizer_config.json',
 '/content/drive/MyDrive/fine_tuned_nmt_model/special_tokens_map.json',
 '/content/drive/MyDrive/fine_tuned_nmt_model/vocab.txt',
 '/content/drive/MyDrive/fine_tuned_nmt_model/added_tokens.json',
 '/content/drive/MyDrive/fine_tuned_nmt_model/tokenizer.json')

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
save_path = '/content/drive/MyDrive/fine_tuned_nmt_model'
os.makedirs(save_path, exist_ok=True)

In [16]:
dataset['train']['translation'][10]['en']

'The love Christ displayed was central to his accomplishing what God has purposed for mankind .'

In [32]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

# Load the fine-tuned model and tokenizer
model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/fine_tuned_nmt_model")
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/fine_tuned_nmt_model")

# Input text to translate
input_text = f"Translate English(en) to South Azerbaijani(azb): how are you today?"

# Tokenize the input
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# Generate a translation
output = model.generate(input_ids)
translation = tokenizer.decode(output[0], skip_special_tokens=True)

print("Translation:", translation)


Translation: بیز اونلاری یيهووانین یيهووانین یيه


In [20]:
input_text

'Translate English(en) to South Azerbaijani(azb): Yes , He surely is Able to do all things .'

In [3]:
!ls /content/drive/MyDrive/fine_tuned_nmt_model

config.json		model.safetensors	 tokenizer_config.json	vocab.txt
generation_config.json	special_tokens_map.json  tokenizer.json
